# Learning in Public: Building a RAG Bot for LLM Zoomcamp

As part of my journey through the DataTalks.Club LLM Zoomcamp, I'm practicing the "learning in public" approach. Instead of keeping my notes tucked away in a local folder, I'm sharing my execution steps, code snippets, and mental models as I build out our running project: a question-answering bot tailored specifically for the course itself. Here is a breakdown of my first hands-on steps with LLMs and the core mechanics of Retrieval-Augmented Generation (RAG).

## Table of Contents
1. Running Project Overview
2. Why RAG is Needed (The Problem)
3. LLM Limitations & Hallucination
4. Adding Context Manually (The Solution)
5. Deconstructing the RAG Architecture
6. Component 1: Search
7. Component 2: Prompt Engineering
8. Component 3: LLM
9. Building the Complete RAG Pipeline
10. Key Takeaways

## Prerequisites

To follow along with this tutorial, you'll need:
- Basic understanding of Python
- Familiarity with API calls and JSON
- OpenAI API key (for gpt-5.4-mini)
- Libraries: `python-dotenv`, `openai`, `minsearch`, `requests` 

Install the required packages using:
```bash
pip install python-dotenv openai minsearch requests
```

## Running Project Overview

In this course, our running project is to build a bot that can answer queries related to the course.

DataTalks.Club runs free ZoomCamps on data engineering, machine learning, MLOps, and other topics. Each course has its own FAQ document containing common questions and answers. Many of these questions overlap across different tracks, but they might be worded differently, or the responses might be structured differently. Additionally, some questions are highly course-specific.

We need to build a bot that can ingest all of this knowledge and provide accurate answers in natural language.

In [59]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

# Reuse the existing client if it was already created in another cell
if "openai_client" not in globals():
    openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [60]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

Let's test how it responds to a general greeting:

In [61]:
llm("Hey Mr. LLM, how is life treating you?")

'Pretty well, thanks for asking. I don’t experience life the way people do, but I’m here and ready to help.\n\nHow’s life treating you?'

Now, let's see how it handles a course-specific question:

In [62]:
question = "I just discovered the course. Can I still join?"
llm_response = llm(question)
print(f"LLM Response: {llm_response}")

LLM Response: Yes—often you can still join, but it depends on the course and whether registration is still open.

A few quick things to check:
- **Enrollment deadline:** Is the signup period still open?
- **Waitlist:** If the course is full, you may still be able to join the waitlist.
- **Late registration policy:** Some courses allow late adds with instructor approval.
- **Prerequisites / access:** Make sure you can still access the platform/materials.

If you want, I can help you draft a short message to the instructor or support team asking if late enrollment is possible.


### Observation

It's evident that LLMs would rarely say, "I don't know the answer to your question." Instead, the model provides a vague, generic response.

If we ask it "How do I make Biryani?"—which is a common knowledge question (although making a good Biryani is still very difficult!)—LLMs can answer beautifully. But because they lack specific knowledge about LLM Zoomcamp, they give vague, hallucinated answers.

### Why does this happen?

An LLM is trained on publicly available data up to a specific knowledge cutoff point. If an LLM's training data cuts off on April 30, 2026, and a new event happens on May 1, 2026, the LLM would have no clue about it. More importantly, it doesn't have access to private, internal, or highly niche documents—like our course FAQs.

This is why LLMs hallucinate: they fill in gaps with plausible-sounding but potentially inaccurate information.

## Adding Context Manually

We can fix this shortcoming by manually providing the necessary context to the LLM. Let's pull some actual FAQ data from the LLM Zoomcamp website and pass it along with our query.

In [63]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

Now, let's construct a structured prompt that instructs the LLM to behave strictly as an assistant grounded in this data:

In [64]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [65]:
answer = llm(prompt)
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.


### Result

Notice how the LLM now provides a grounded answer based on the context we provided. This is the essence of RAG—we're augmenting the LLM's knowledge with specific, relevant information from our knowledge base.

## Deconstructing the RAG Architecture

RAG stands for **Retrieval-Augmented Generation**.

Instead of relying solely on the LLM's internal weights, we retrieve relevant documents from our knowledge base and augment the prompt to give the model the precise context it needs to generate an accurate answer.

Ultimately, the entire architecture boils down to three core pillars:

- **Search**: Finding relevant documents/information from the knowledge base
- **Prompt**: Building a structured prompt with the query and retrieved context
- **LLM**: Generating the final answer based on grounded information

### RAG Architecture Diagram

```mermaid
flowchart TD
    U([User])
    APP[Application]
    DB[(Knowledge Base)]
    DOCS[[D1...D5]]
    PROMPT[Build Prompt<br/>Question + Context]
    LLM[LLM]
    ANSWER([Answer])

    U -->|Question| APP
    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS
    DOCS --> APP
    APP --> PROMPT
    PROMPT --> LLM
    LLM --> ANSWER
    ANSWER --> U
```

### Key Insight

Because the LLM only sees the data we explicitly hand to it, its answers are entirely grounded. If we retrieve the right data, the answer is spot-on. If we retrieve the wrong data, the answer degrades.

**Our model is only as good as our retrieval engine.**

This is why search quality is the most critical factor in production-grade RAG pipelines.

## Component 1: Search

### How Search Works

At the core, every search engine does the same thing:
1. Take a query
2. Evaluate similarity between the query and each document and score every document
3. Return the top results

We can consider the **Search** as a similarity function. What differentiates every search engine is:
- What similarity metric is used?
- How is similarity calculated?

### Types of Search

Broadly, there can be 2 kinds of similarity:

**Text/Lexical Search**: 
- Counts how many words the query and the document share
- Looks at similarity only at the surface level

**Vector/Semantic Search**: 
- Compares the meaning of the documents and the query
- Goes deeper to assess similarity based on semantic understanding

### Example: The Difference Between Lexical and Semantic Search

Consider these two questions:
- "Can I still join the course after the start date?"
- "Is it possible to enroll late?"

They mean the same thing, but share almost no keywords:
- "Join" is not "enroll"
- "course" is absent
- "start date" is not "late"

A text search engine would struggle to match them because it only sees words. Vector search, which we'll cover later in the course, would understand the semantic similarity.

For now, we'll use text-based search implemented with **minsearch**, a lightweight search engine that can be used with Python.

### Indexing with minsearch

It is very important to index documents for efficient searching. If we don't index and send all the documents to the LLM, it would be very expensive and slow.

[minsearch](https://github.com/alexeygrigorev/minsearch) is a simple in-memory search engine. It's lightweight, so it runs anywhere Python runs. It's a toy implementation, not production-ready, but it illustrates how search engines work and gives good results.

We'll index the question, section, and answer fields as text (they'll be tokenized and ranked), and the course field as a keyword (for filtering).

In [66]:
from minsearch import Index ## used to create an index for the FAQ data
import requests
# Load the FAQ data
def load_faq_data():
    docs_url = 'https://datatalks.club/faq/json/courses.json'
    response = requests.get(docs_url)
    courses_raw = response.json()

    documents = []
    url_prefix = 'https://datatalks.club/faq'

    for course in courses_raw:
        course_url = f'{url_prefix}{course["path"]}'
        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()

        documents.extend(course_data)

    return documents
documents = load_faq_data()
# Initialize the index with text and keyword fields
index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

# Fit the index with documents
index.fit(documents)

### Trying a Search

Let's try a search with the question we used before:

In [67]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

print(search_results)

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '977bf7786c', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?', 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}, {'id': '69d122f12e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?', 'answer': 'No, you can only get a certificate 

### Boosting Fields

Not all fields are equally important. The question field is usually more relevant than section for matching. Query words appearing in the question is a stronger signal than them appearing in the section name.

minsearch supports field boosting to reflect this importance:

```python
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)
```

In this example:
- Matches in the "question" field are weighted 2x
- Matches in the "section" field are weighted 0.5x (less important)
- Matches in other text fields get a default weight of 1.0

### Filtering by Course

We want to restrict the search to a specific course to ensure we get only relevant results for that course.

minsearch supports keyword filtering:

```python
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "llm-zoomcamp"}
)
```

This restricts results to only documents where the course field matches "llm-zoomcamp".

### Wrapping it in a Function

Let's wrap the search in a search function - the first component of our RAG pipeline:

In [68]:
def search(question, course="llm-zoomcamp"):
    """Search for relevant documents in the index."""
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

# Example usage
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

## Component 2: Prompt Engineering

With **Search**, we now have a relevant Knowledge Base and can retrieve specific documents. However, the LLM won't see this information unless we explicitly pass it along. So the prompt must include both the **user's question** and the **search results**.

### Anatomy of a Good Prompt

The Prompt consists of two distinct parts:
- **System Instructions** (System Prompt): Tells the LLM how to behave and sets its role. These instructions remain constant across all requests, ensuring consistent behavior.
- **User Prompt**: Changes with each question. It carries the actual user query and the retrieved context from the knowledge base.

### System Instructions

The instructions define the LLMs role and answer format. This grounding is crucial for reducing hallucinations and ensuring answers stay within the knowledge base.

In [69]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

### User Prompt Template

The user prompt template has placeholders for the question and context, making it easy to swap in new queries:

In [70]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

### Building the Context

The **context** is a formatted string aggregating all search results into a readable format. Each document becomes a structured block with the section, question, and answer. This preprocessing transforms a list of dictionaries into clean, readable text that the LLM can easily parse and reference.

In [71]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

### Building the Prompt

We combine the question and context into the final **user prompt**. This becomes the second message in our conversation:

In [72]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question, 
        context=context)
    return prompt.strip()

prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

### The Power of Good Prompts

The **prompt** is the link between **Search** and the **LLM**. A well-crafted prompt keeps the LLM grounded in truth, while a poor prompt lets it roam freely and hallucinate.

Prompt engineering is part science and part **art**. You'll need to experiment with different formats, instructions, and context structures to see what works best for your use case.

## Component 3: LLM

This is the final component of our RAG pipeline. It takes the structured prompt we built and generates an answer grounded in the retrieved context.

We send our prompt to the LLM using OpenAI's Responses API:

In [73]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)
response.output

[ResponseOutputMessage(id='msg_0ff9bd0cf9f70145006a537935b56c8195a265236b8b8059c1', content=[ResponseOutputText(annotations=[], text='Yes — you can join now.\n\nYou can start learning and submitting homework right away while the submission form is open. If you want a certificate, make sure you submit your project while the course is still accepting submissions, and note that certificates are only available for the live cohort, not self-paced mode.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

The message has a content list, and the text is in the first item:

In [74]:
print(response.output[0].content[0].text)

Yes — you can join now.

You can start learning and submitting homework right away while the submission form is open. If you want a certificate, make sure you submit your project while the course is still accepting submissions, and note that certificates are only available for the live cohort, not self-paced mode.


That's quite a journey to reach one string.

The shortcut spares us all of it:
```python
response.output_text
```

In [75]:
print(response.output_text)

Yes — you can join now.

You can start learning and submitting homework right away while the submission form is open. If you want a certificate, make sure you submit your project while the course is still accepting submissions, and note that certificates are only available for the live cohort, not self-paced mode.


### Message History: Multi-Turn Conversations (Optional Advanced Topic)

In this tutorial, we send one question and receive one response. However, real-world chat systems maintain a conversation history—a sequence of messages where each has a role (system, user, assistant).

Think of ChatGPT: It starts with a system prompt (hidden from users) that sets behavior guidelines. Then users and the assistant exchange messages. Since LLMs have no inherent memory, the entire conversation history must be appended to each new prompt.

In our simplified implementation, we separate concerns by using different message roles to structure the conversation:

We send two messages:
- `developer` - System-level instructions defining LLM behavior and role
- `user` - The actual prompt containing the question and retrieved context

```python
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)
```

This approach separates static `INSTRUCTIONS` (which remain constant) from the dynamic `user_prompt` (which changes with each query), keeping our system flexible and maintainable.

### Building the LLM Function

We consolidate everything into an updated `llm()` function that handles the complete interaction. It accepts both system instructions and the user prompt:

In [77]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

## Building the Complete RAG Pipeline

Now we integrate all three components into a unified RAG system:

**The Three Core Components:**
1. **Search**: Retrieves relevant documents from the knowledge base using similarity matching
2. **Prompt**: Structures the user query with retrieved context for the LLM
3. **LLM**: Generates grounded answers using only the provided context

**The Complete Pipeline Flow:**
1. User submits a question
2. Search engine retrieves the top matching documents from the knowledge base
3. Retrieved documents are formatted into a context string
4. Prompt is built combining user question + context
5. Prompt is sent to the LLM
6. LLM generates a grounded answer
7. Answer is returned to the user

In [79]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join now. If you want a certificate, you need to submit your project while submissions are still open.


### The revised Architecture
```mermaid
flowchart TD
    U([User])

    APP[Application]

    DB[(Knowledge Base)]
    DOCS[[D1 ... D5]]

    PROMPT[Build Prompt<br/>Question + Context]

    LLM[LLM]

    ANSWER([Answer])

    U -->|Question| APP

    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS
    DOCS --> APP

    APP --> PROMPT
    PROMPT --> LLM

    LLM --> ANSWER
    ANSWER --> U
```



## Key Takeaways

### What is RAG and why does it matter?

Retrieval-Augmented Generation (RAG) elegantly solves the LLM hallucination problem:

1. **Retrieval** → Search knowledge base for relevant information
2. **Augmentation** → Include that information in the prompt
3. **Generation** → LLM generates answers based *only* on provided context

**The Result:** Grounded answers rooted in actual knowledge, not plausible guesses.

### Critical Insight

The quality of your RAG system is limited by the quality of your retrieval component. Even the best LLM cannot work with irrelevant or incorrect search results.

Therefore:
- **Search quality is paramount** in production RAG pipelines
- A mediocre LLM with excellent search beats an excellent LLM with poor search
- Optimizing search algorithms (via field boosting, filtering, better embeddings) has outsized impact

### What We Built

- A function that takes a user question
- Searches for relevant FAQ documents using text similarity
- Constructs a grounded prompt with system instructions
- Sends it to an LLM
- Returns a factual, contextual answer

This simple pattern scales to power production Q&A systems for complex knowledge bases.

---

**Next Steps:** In upcoming posts, we'll explore vector/semantic search, implement better relevance ranking, handle edge cases, and scale this pattern to production systems.